# Heimdall Anomaly Detection - LoRA Training
Model: Qwen/Qwen2.5-0.5B-Instruct
Dataset: Kaggle logging-and-monitoring-anomalies (downloaded via kagglehub)

In [ ]:
# Install PyTorch 2.0.1 with CUDA 11.7 for P100 (sm_60) support
%pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117
%pip install -q kagglehub unsloth transformers datasets trl peft accelerate bitsandbytes

In [ ]:
# Download dataset via kagglehub (works in Kaggle kernels)
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download('mirzayasirabdullah07/logging-and-monitoring-anomalies-dataset')
print(f"Downloaded to: {path}")

# Find the CSV file
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.csv'):
            csv_path = os.path.join(root, f)
            print(f"Found CSV: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Total rows: {len(df)}")
print(df.columns.tolist())
print(df['Severity'].value_counts())

In [ ]:
# Convert to JSONL format for training
import json
os.makedirs('/kaggle/working/data', exist_ok=True)

def create_messages(row):
    severity = row['Severity']
    label = "ANOMALOUS" if severity in ['High', 'Critical'] else "NORMAL"
    
    system_prompt = "You are a security surveillance expert. Classify this system log event as ANOMALOUS or NORMAL. Focus on file system changes, log patterns indicating compromise, or suspicious activity."
    
    user_content = f"File system event detected:\nSource: {row['Source']}\nType: {row['Anomaly_Type']}\nSeverity: {severity}\nProcess ID: {row['Process_ID']}\nHost IP: {row['Host_IP']}\nError Code: {row['Error_Code']}\nAffected Services: {row['Affected_Services']}\nCPU/Memory/Disk: {row['CPU_Usage_Percent']}%/{row['Memory_Usage_MB']}MB/{row['Disk_Usage_Percent']}%"
    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": f"{label}. {row['Anomaly_Type']} activity detected."}
        ]
    }

# Split 90/10
train_size = int(len(df) * 0.9)
train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]

with open('/kaggle/working/data/train.jsonl', 'w') as f:
    for _, row in train_df.iterrows():
        f.write(json.dumps(create_messages(row)) + '\n')

with open('/kaggle/working/data/test.jsonl', 'w') as f:
    for _, row in test_df.iterrows():
        f.write(json.dumps(create_messages(row)) + '\n')

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

In [ ]:
# Load dataset for training
from datasets import load_dataset
dataset = load_dataset('json', data_files={'train': '/kaggle/working/data/train.jsonl', 'test': '/kaggle/working/data/test.jsonl'})
print(f"Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

In [ ]:
# Load model with 4-bit quantization
from unsloth import FastLanguageModel
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
)

In [ ]:
# LoRA configuration
lora_config = {
    'r': 64,
    'lora_alpha': 128,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    'lora_dropout': 0.05,
}

model = FastLanguageModel.get_peft_model(model, lora_config)
print('LoRA applied')

In [ ]:
# Training
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    dataset_text_field = 'messages',
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 20,
        max_steps = 200,
        learning_rate = 2e-5,
        fp16 = True,
        logging_steps = 20,
        output_dir = '/kaggle/working/outputs',
        report_to = "none",
    ),
)

trainer.train()